# Ceres Raw Integrity, p128 Patch Factory, and Kaggle Dataset Pipeline

This notebook is the integrated Colab workflow for the 2025W36-W52 p128 season dataset.

It does three jobs in order:

1. Scan all raw GeoTIFFs in Drive and write integrity reports back to Drive.
2. Generate p128 sharded patch data for 2025W36-W52 directly into Drive.
3. Optionally upload the staged p128 folder as a private Kaggle Dataset.

This notebook intentionally avoids large `/content` staging folders. Shards are written to Drive as `.tmp.npz` and then renamed to `.npz`.


In [ ]:
# Mount Drive and install runtime dependencies.
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Not running in Colab; skipping drive.mount().")

import subprocess
import sys

missing = []
try:
    import rasterio  # noqa: F401
except ModuleNotFoundError:
    missing.append("rasterio")

try:
    import tqdm  # noqa: F401
except ModuleNotFoundError:
    missing.append("tqdm")

try:
    import pandas  # noqa: F401
except ModuleNotFoundError:
    missing.append("pandas")

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


In [ ]:
from __future__ import annotations

import csv
import json
import math
import os
import re
import shutil
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from tqdm.auto import tqdm


## Configuration

Defaults are set for the current Drive layout. `SCAN_WEEK_START` and `SCAN_WEEK_END` are `None` so the integrity scan checks every matching raw file in the folder. Patch generation still uses `WEEK_START` to `WEEK_END`.


In [ ]:
DRIVE_ROOT = Path("/content/drive/MyDrive")

RAW_DIR = DRIVE_ROOT / "wheat_data_v2.0-beta"
REPORT_DIR = DRIVE_ROOT / "Ceres" / "raw_integrity_reports" / "wheat_data_v2.0-beta"
OUTPUT_ROOT = DRIVE_ROOT / "Ceres" / "staged" / "2025w36_w52_grid_v1_p128"

SCAN_WEEK_START = None
SCAN_WEEK_END = None

WEEK_START = "2025W36"
WEEK_END = "2025W52"
PATCH_SIZES = (128,)
STRIPE_HEIGHT = 256
VALID_RATIO_MIN = 0.80
NODATA = -32768.0
SKIP_EXISTING = True
FAIL_ON_BAD_RAW = True

FEATURE_BANDS = (
    "ndvi",
    "ndmi",
    "nbr",
    "s1_vv",
    "s1_vh",
    "s1_vh_vv",
    "rain_mm",
    "temp_c_mean",
    "temp_c_max",
    "lst_c",
)
LABEL_BAND = "risk"
FINAL_BANDS = FEATURE_BANDS + (LABEL_BAND,)

print("RAW_DIR", RAW_DIR)
print("REPORT_DIR", REPORT_DIR)
print("OUTPUT_ROOT", OUTPUT_ROOT)


In [ ]:
RAW_NAME_RE = re.compile(
    r"^fr_wheat_feat_(?P<year>\d{4})W(?P<week>\d{2})"
    r"(?:-(?P<row>\d+)-(?P<col>\d+))?\.tif(?:f)?$",
    re.IGNORECASE,
)

@dataclass(frozen=True, slots=True)
class RawTile:
    path: Path
    week_key: str
    year: int
    week: int
    row_offset: int
    col_offset: int
    tile_key: str
    tile_id: int
    split: str
    width: int
    height: int
    count: int
    crs: str
    nodata: float | None
    dtypes: tuple[str, ...]
    descriptions: tuple[str, ...]

def utc_now() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

def parse_week_key(week_key: str) -> tuple[int, int]:
    m = re.match(r"^(\d{4})W(\d{2})$", week_key, re.IGNORECASE)
    if not m:
        raise ValueError(f"Invalid week key: {week_key}")
    return int(m.group(1)), int(m.group(2))

def in_week_range(week_key: str, start: str | None, end: str | None) -> bool:
    wk = parse_week_key(week_key)
    if start is not None and wk < parse_week_key(start):
        return False
    if end is not None and wk > parse_week_key(end):
        return False
    return True

def parse_raw_name(path: Path) -> tuple[str, int, int, int, int, str] | None:
    m = RAW_NAME_RE.match(path.name)
    if not m:
        return None
    year = int(m.group("year"))
    week = int(m.group("week"))
    row_offset = int(m.group("row") or 0)
    col_offset = int(m.group("col") or 0)
    week_key = f"{year}W{week:02d}"
    tile_key = f"r{row_offset:010d}_c{col_offset:010d}"
    return week_key, year, week, row_offset, col_offset, tile_key

def normalize_descriptions(descriptions: Iterable[str | None]) -> tuple[str, ...]:
    return tuple((d or "").strip() for d in descriptions)

def file_magic(path: Path, n: int = 16) -> str:
    try:
        with path.open("rb") as f:
            return repr(f.read(n))
    except Exception as e:
        return f"READ_FAILED: {e}"

def append_csv(path: Path, row: dict[str, object], fieldnames: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    exists = path.exists()
    with path.open("a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        if not exists:
            writer.writeheader()
        writer.writerow(row)

def write_csv_rows(path: Path, rows: list[dict[str, object]], fieldnames: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    with tmp_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
    tmp_path.replace(path)

def write_json(path: Path, value: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    tmp_path.write_text(json.dumps(value, indent=2, sort_keys=True) + "\n")
    tmp_path.replace(path)


## Raw Integrity Scan

This scan reads raw file headers and metadata only. It does not copy raw GeoTIFFs, does not create patches, and writes only small CSV/JSON reports to Drive.


In [ ]:
if True:
    GOOD_FIELDS = [
        "path", "filename", "size_bytes", "magic",
        "year", "week", "week_key", "row_offset", "col_offset", "tile_key",
        "status", "issues", "width", "height", "count", "crs", "nodata",
        "dtypes", "descriptions",
    ]
    BAD_FIELDS = [
        "path", "filename", "size_bytes", "magic",
        "year", "week", "week_key", "row_offset", "col_offset", "tile_key",
        "status", "error", "issues",
    ]
    UNMATCHED_FIELDS = ["path", "filename", "size_bytes", "magic", "status"]

    def scan_raw_integrity() -> dict[str, object]:
        REPORT_DIR.mkdir(parents=True, exist_ok=True)
        good_path = REPORT_DIR / "good_files.csv"
        bad_path = REPORT_DIR / "bad_files.csv"
        unmatched_path = REPORT_DIR / "unmatched_files.csv"
        summary_path = REPORT_DIR / "summary.json"

        files = sorted(p for p in RAW_DIR.glob("*.tif*") if p.is_file())
        good_count = 0
        bad_count = 0
        unmatched_count = 0
        weeks_good: set[str] = set()
        weeks_bad: set[str] = set()
        first_bad: list[dict[str, object]] = []
        good_rows: list[dict[str, object]] = []
        bad_rows: list[dict[str, object]] = []
        unmatched_rows: list[dict[str, object]] = []

        print("RAW_DIR:", RAW_DIR)
        print("REPORT_DIR:", REPORT_DIR)
        print("candidate files:", len(files))

        for p in tqdm(files, desc="Scanning raw GeoTIFFs", unit="file"):
            parsed = parse_raw_name(p)
            size = p.stat().st_size
            magic = file_magic(p)
            base = {
                "path": str(p),
                "filename": p.name,
                "size_bytes": size,
                "magic": magic,
            }

            if parsed is None:
                unmatched_count += 1
                unmatched_rows.append({**base, "status": "unmatched_filename"})
                continue

            week_key, year, week, row_offset, col_offset, tile_key = parsed
            if not in_week_range(week_key, SCAN_WEEK_START, SCAN_WEEK_END):
                continue

            parsed_row = {
                "year": year,
                "week": week,
                "week_key": week_key,
                "row_offset": row_offset,
                "col_offset": col_offset,
                "tile_key": tile_key,
            }

            try:
                with rasterio.open(p) as ds:
                    descriptions = normalize_descriptions(ds.descriptions)
                    dtypes = tuple(str(d) for d in ds.dtypes)
                    issues: list[str] = []

                    if ds.count != len(FINAL_BANDS):
                        issues.append(f"band_count={ds.count}")
                    if descriptions != FINAL_BANDS:
                        issues.append(f"band_order={descriptions}")
                    if any(dtype.lower() != "float32" for dtype in dtypes):
                        issues.append(f"dtypes={dtypes}")
                    if ds.nodata is None or float(ds.nodata) != NODATA:
                        issues.append(f"nodata={ds.nodata}")
                    if ds.crs is None:
                        issues.append("missing_crs")

                    row = {
                        **base,
                        **parsed_row,
                        "status": "ok" if not issues else "schema_issue",
                        "issues": ";".join(issues),
                        "width": ds.width,
                        "height": ds.height,
                        "count": ds.count,
                        "crs": str(ds.crs),
                        "nodata": ds.nodata,
                        "dtypes": "|".join(dtypes),
                        "descriptions": "|".join(descriptions),
                    }

                    if issues:
                        bad_count += 1
                        weeks_bad.add(week_key)
                        bad_rows.append(row)
                        if len(first_bad) < 30:
                            first_bad.append(row)
                    else:
                        good_count += 1
                        weeks_good.add(week_key)
                        good_rows.append(row)

            except Exception as e:
                row = {
                    **base,
                    **parsed_row,
                    "status": "open_failed",
                    "error": str(e),
                    "issues": "",
                }
                bad_count += 1
                weeks_bad.add(week_key)
                bad_rows.append(row)
                if len(first_bad) < 30:
                    first_bad.append(row)

        write_csv_rows(good_path, good_rows, GOOD_FIELDS)
        write_csv_rows(bad_path, bad_rows, BAD_FIELDS)
        write_csv_rows(unmatched_path, unmatched_rows, UNMATCHED_FIELDS)

        report = {
            "created_at": utc_now(),
            "raw_dir": str(RAW_DIR),
            "total_files": len(files),
            "good_count": good_count,
            "bad_count": bad_count,
            "unmatched_count": unmatched_count,
            "weeks_good": sorted(weeks_good),
            "weeks_bad": sorted(weeks_bad),
        }
        write_json(summary_path, report)

        print(json.dumps(report, indent=2, sort_keys=True))
        print("\nFirst 30 bad files:")
        for row in first_bad:
            print(
                row.get("week_key"),
                row["filename"],
                row["status"],
                row.get("size_bytes"),
                row.get("magic"),
                row.get("error", row.get("issues", "")),
            )
        print("\nReports written to:", REPORT_DIR)
        return report


In [ ]:
integrity_report = scan_raw_integrity()
integrity_report


## Patch Factory Helpers

Patch generation reads selected valid raw files and writes p128 shard files directly to Drive. Existing valid shards are skipped when `SKIP_EXISTING=True`.


In [ ]:
def validate_dimensions_for_patching(
    *,
    filename: str,
    width: int,
    height: int,
    patch_sizes: Iterable[int],
    stripe_height: int,
) -> None:
    for patch_size in patch_sizes:
        if width < patch_size or height < patch_size:
            raise RuntimeError(
                f"{filename}: dimensions {width}x{height} must be >= patch_size={patch_size}"
            )
    if stripe_height <= 0:
        raise RuntimeError(f"STRIPE_HEIGHT must be > 0, got {stripe_height}")

def validate_raster(path: Path) -> dict[str, object]:
    with rasterio.open(path) as ds:
        descriptions = normalize_descriptions(ds.descriptions)
        if ds.count != len(FINAL_BANDS):
            raise RuntimeError(f"{path.name}: expected {len(FINAL_BANDS)} bands, got {ds.count}")
        if descriptions != FINAL_BANDS:
            raise RuntimeError(f"{path.name}: unexpected band descriptions {descriptions}; expected {FINAL_BANDS}")
        if any(str(dtype).lower() != "float32" for dtype in ds.dtypes):
            raise RuntimeError(f"{path.name}: all bands must be float32, got {ds.dtypes}")
        if ds.nodata is None or not math.isclose(float(ds.nodata), NODATA):
            raise RuntimeError(f"{path.name}: expected nodata={NODATA}, got {ds.nodata}")
        if ds.crs is None:
            raise RuntimeError(f"{path.name}: CRS is required")
        validate_dimensions_for_patching(
            filename=path.name,
            width=int(ds.width),
            height=int(ds.height),
            patch_sizes=PATCH_SIZES,
            stripe_height=STRIPE_HEIGHT,
        )
        return {
            "width": int(ds.width),
            "height": int(ds.height),
            "count": int(ds.count),
            "crs": str(ds.crs),
            "nodata": float(ds.nodata),
            "dtypes": tuple(str(d) for d in ds.dtypes),
            "descriptions": descriptions,
        }

def build_tile_splits(tile_keys: list[str]) -> dict[str, str]:
    n = len(tile_keys)
    if n == 0:
        raise RuntimeError("No tiles found")
    if n == 1:
        return {tile_keys[0]: "train"}
    if n == 2:
        return {tile_keys[0]: "train", tile_keys[1]: "val"}
    if n == 3:
        return {tile_keys[0]: "train", tile_keys[1]: "val", tile_keys[2]: "test"}

    train_count = max(1, int(n * 0.70))
    val_count = max(1, int(n * 0.15))
    if train_count + val_count >= n:
        train_count = max(1, n - 2)
        val_count = 1

    out: dict[str, str] = {}
    for i, key in enumerate(tile_keys):
        if i < train_count:
            out[key] = "train"
        elif i < train_count + val_count:
            out[key] = "val"
        else:
            out[key] = "test"
    return out

def scan_raw_tiles(raw_dir: Path) -> list[RawTile]:
    if not raw_dir.exists():
        raise FileNotFoundError(f"RAW_DIR does not exist: {raw_dir}")

    candidates = sorted(
        p for p in raw_dir.rglob("*.tif*") if p.is_file() and parse_raw_name(p) is not None
    )
    parsed = [(p, parse_raw_name(p)) for p in candidates]
    parsed = [item for item in parsed if item[1] is not None]
    parsed = [
        (p, info)
        for p, info in parsed
        if in_week_range(info[0], WEEK_START, WEEK_END)
    ]
    if not parsed:
        raise RuntimeError(f"No raw GeoTIFFs found in {raw_dir} for {WEEK_START}..{WEEK_END}")

    tile_keys = sorted({info[5] for _p, info in parsed})
    tile_id_by_key = {key: i for i, key in enumerate(tile_keys)}
    split_by_key = build_tile_splits(tile_keys)

    tiles: list[RawTile] = []
    for path, info in tqdm(parsed, desc="Validating raw GeoTIFFs", unit="file"):
        week_key, year, week, row_offset, col_offset, tile_key = info
        meta = validate_raster(path)
        tiles.append(
            RawTile(
                path=path,
                week_key=week_key,
                year=year,
                week=week,
                row_offset=row_offset,
                col_offset=col_offset,
                tile_key=tile_key,
                tile_id=tile_id_by_key[tile_key],
                split=split_by_key[tile_key],
                width=int(meta["width"]),
                height=int(meta["height"]),
                count=int(meta["count"]),
                crs=str(meta["crs"]),
                nodata=float(meta["nodata"]),
                dtypes=tuple(meta["dtypes"]),
                descriptions=tuple(meta["descriptions"]),
            )
        )
    tiles.sort(key=lambda t: (parse_week_key(t.week_key), t.row_offset, t.col_offset, t.path.name))
    return tiles


In [ ]:
def write_inventory(output_root: Path, tiles: list[RawTile]) -> None:
    output_root.mkdir(parents=True, exist_ok=True)
    path = output_root / "raw_inventory.csv"
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "path", "filename", "week_key", "tile_key", "tile_id", "split",
                "row_offset", "col_offset", "width", "height", "count", "crs", "nodata",
            ],
        )
        writer.writeheader()
        for tile in tiles:
            writer.writerow({
                "path": str(tile.path),
                "filename": tile.path.name,
                "week_key": tile.week_key,
                "tile_key": tile.tile_key,
                "tile_id": tile.tile_id,
                "split": tile.split,
                "row_offset": tile.row_offset,
                "col_offset": tile.col_offset,
                "width": tile.width,
                "height": tile.height,
                "count": tile.count,
                "crs": tile.crs,
                "nodata": tile.nodata,
            })
    print("Wrote", path)

def write_split_manifest(output_root: Path, tiles: list[RawTile]) -> None:
    by_tile: dict[str, RawTile] = {}
    for tile in tiles:
        by_tile.setdefault(tile.tile_key, tile)
    manifest = {
        "schema_version": "tile-split-v1",
        "created_at": utc_now(),
        "split_strategy": "tile_based_offsets_initial",
        "week_start": WEEK_START,
        "week_end": WEEK_END,
        "tile_split": {k: by_tile[k].split for k in sorted(by_tile)},
        "tiles": [
            {
                "tile_key": by_tile[k].tile_key,
                "tile_id": by_tile[k].tile_id,
                "split": by_tile[k].split,
                "row_offset": by_tile[k].row_offset,
                "col_offset": by_tile[k].col_offset,
            }
            for k in sorted(by_tile)
        ],
    }
    write_json(output_root / "split_manifest.json", manifest)
    print("Wrote", output_root / "split_manifest.json")

def shard_name_for(tile: RawTile, row_start: int, row_end_exclusive: int, patch_size: int) -> str:
    return f"{tile.week_key}_{tile.tile_key}_y{row_start:06d}_y{row_end_exclusive - 1:06d}_p{patch_size}.npz"

def valid_final_shard(path: Path) -> bool:
    if not path.exists():
        return False
    try:
        with np.load(path, allow_pickle=False) as z:
            required = {"X", "y", "valid_mask"}
            return required.issubset(set(z.files)) and int(z["y"].shape[0]) > 0
    except Exception:
        return False

def save_npz_atomic(final_path: Path, arrays: dict[str, np.ndarray]) -> None:
    final_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = final_path.with_suffix(".tmp.npz")
    if tmp_path.exists():
        tmp_path.unlink()
    np.savez_compressed(tmp_path, **arrays)
    tmp_path.replace(final_path)


In [ ]:
def build_shard(tile: RawTile, patch_size: int, row_start: int, row_end_exclusive: int) -> dict[str, np.ndarray] | None:
    stripe_height = row_end_exclusive - row_start
    if stripe_height < patch_size:
        return None

    with rasterio.open(tile.path) as ds:
        stripe = ds.read(
            indexes=list(range(1, len(FINAL_BANDS) + 1)),
            window=Window(0, row_start, ds.width, stripe_height),
        ).astype(np.float32, copy=False)

    xs: list[np.ndarray] = []
    ys: list[np.float32] = []
    masks: list[np.ndarray] = []
    rows: list[int] = []
    cols: list[int] = []
    ndvi_band = stripe[0]
    risk_band = stripe[-1]

    for local_row in range(0, stripe_height - patch_size + 1, patch_size):
        for col in range(0, tile.width - patch_size + 1, patch_size):
            row = row_start + local_row
            ndvi = ndvi_band[local_row:local_row + patch_size, col:col + patch_size]
            risk = risk_band[local_row:local_row + patch_size, col:col + patch_size]
            valid_mask = (
                np.isfinite(ndvi)
                & np.isfinite(risk)
                & (ndvi != NODATA)
                & (risk != NODATA)
            )
            if float(valid_mask.mean()) < VALID_RATIO_MIN:
                continue
            y = np.float32(risk[valid_mask].mean())
            x = stripe[:len(FEATURE_BANDS), local_row:local_row + patch_size, col:col + patch_size]
            xs.append(np.array(x, dtype=np.float32, copy=True))
            ys.append(y)
            masks.append(np.array(valid_mask, dtype=bool, copy=True))
            rows.append(row)
            cols.append(col)

    if not xs:
        return None

    n = len(xs)
    return {
        "X": np.stack(xs, axis=0).astype(np.float32, copy=False),
        "y": np.asarray(ys, dtype=np.float32),
        "valid_mask": np.stack(masks, axis=0).astype(bool, copy=False),
        "week_key": np.asarray([tile.week_key] * n),
        "tile_key": np.asarray([tile.tile_key] * n),
        "tile_id": np.full((n,), tile.tile_id, dtype=np.int16),
        "split": np.asarray([tile.split] * n),
        "row": np.asarray(rows, dtype=np.int32),
        "col": np.asarray(cols, dtype=np.int32),
        "stripe_row_start": np.asarray(row_start, dtype=np.int32),
        "stripe_row_end": np.asarray(row_end_exclusive - 1, dtype=np.int32),
        "row_offset": np.full((n,), tile.row_offset, dtype=np.int32),
        "col_offset": np.full((n,), tile.col_offset, dtype=np.int32),
        "patch_size": np.asarray(patch_size, dtype=np.int16),
        "nodata": np.asarray(NODATA, dtype=np.float32),
    }

def generate_shards_for_patch_size(tiles: list[RawTile], patch_size: int) -> None:
    dataset_dir = OUTPUT_ROOT / f"p{patch_size}"
    shards_dir = dataset_dir / "shards"
    shards_dir.mkdir(parents=True, exist_ok=True)

    total_jobs = sum(math.ceil(tile.height / STRIPE_HEIGHT) for tile in tiles)
    kept_shards = 0
    kept_samples = 0
    skipped_existing = 0

    with tqdm(total=total_jobs, desc=f"Generating p{patch_size} shards", unit="stripe") as bar:
        for tile in tiles:
            for row_start in range(0, tile.height, STRIPE_HEIGHT):
                row_end = min(tile.height, row_start + STRIPE_HEIGHT)
                final_path = shards_dir / shard_name_for(tile, row_start, row_end, patch_size)

                if SKIP_EXISTING and valid_final_shard(final_path):
                    with np.load(final_path, allow_pickle=False) as z:
                        kept_samples += int(z["y"].shape[0])
                    kept_shards += 1
                    skipped_existing += 1
                    bar.update(1)
                    continue

                arrays = build_shard(tile, patch_size, row_start, row_end)
                if arrays is not None:
                    save_npz_atomic(final_path, arrays)
                    kept_shards += 1
                    kept_samples += int(arrays["y"].shape[0])
                bar.update(1)

    print(f"p{patch_size}: wrote/kept {kept_shards} shards, {kept_samples} samples, skipped_existing={skipped_existing}")


In [ ]:
def read_shard_summary(dataset_dir: Path, shard_path: Path) -> dict[str, object]:
    with np.load(shard_path, allow_pickle=False) as z:
        y = z["y"]
        patch_size = int(np.asarray(z["patch_size"]).item())
        rel = shard_path.relative_to(dataset_dir).as_posix()
        if "stripe_row_start" in z.files:
            row_start = int(np.asarray(z["stripe_row_start"]).item())
            row_end = int(np.asarray(z["stripe_row_end"]).item())
        else:
            rows = z["row"]
            row_start = int(rows.min())
            row_end = int(rows.max()) + patch_size - 1
        return {
            "shard_path": rel,
            "patch_size": patch_size,
            "num_samples": int(y.shape[0]),
            "week_key": str(z["week_key"][0]),
            "tile_key": str(z["tile_key"][0]),
            "tile_id": int(z["tile_id"][0]),
            "split": str(z["split"][0]),
            "row_start": row_start,
            "row_end": row_end,
            "col_offset": int(z["col_offset"][0]),
        }

def rebuild_index_and_manifest(tiles: list[RawTile], patch_size: int) -> None:
    dataset_dir = OUTPUT_ROOT / f"p{patch_size}"
    shards = sorted((dataset_dir / "shards").glob("*.npz"))
    summaries = [read_shard_summary(dataset_dir, p) for p in shards]
    summaries = [s for s in summaries if int(s["num_samples"]) > 0]
    summaries.sort(key=lambda r: (r["week_key"], r["tile_id"], r["row_start"], r["shard_path"]))

    index_path = dataset_dir / "index.csv"
    fieldnames = [
        "shard_path", "patch_size", "num_samples", "week_key", "tile_key",
        "tile_id", "split", "row_start", "row_end", "col_offset",
    ]
    with index_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(summaries)

    split_counts: dict[str, int] = {}
    week_counts: dict[str, int] = {}
    for row in summaries:
        split = str(row["split"])
        week_key = str(row["week_key"])
        n = int(row["num_samples"])
        split_counts[split] = split_counts.get(split, 0) + n
        week_counts[week_key] = week_counts.get(week_key, 0) + n

    manifest = {
        "schema_version": "ceres-sharded-npz-v1",
        "created_at": utc_now(),
        "raw_dir": str(RAW_DIR),
        "output_dir": str(dataset_dir),
        "week_start": WEEK_START,
        "week_end": WEEK_END,
        "patch_size": patch_size,
        "stripe_height": STRIPE_HEIGHT,
        "feature_bands": list(FEATURE_BANDS),
        "label_band": LABEL_BAND,
        "final_bands": list(FINAL_BANDS),
        "nodata": NODATA,
        "valid_ratio_min": VALID_RATIO_MIN,
        "valid_mask_definition": "(ndvi != nodata) AND (risk != nodata)",
        "raw_file_count": len(tiles),
        "total_shards": len(summaries),
        "total_samples": sum(int(r["num_samples"]) for r in summaries),
        "split_counts": split_counts,
        "week_counts": week_counts,
        "index_csv": "index.csv",
    }
    write_json(dataset_dir / "manifest.json", manifest)
    print("Wrote", index_path)
    print("Wrote", dataset_dir / "manifest.json")

def smoke_load(patch_size: int) -> None:
    dataset_dir = OUTPUT_ROOT / f"p{patch_size}"
    index_path = dataset_dir / "index.csv"
    with index_path.open(newline="") as f:
        rows = list(csv.DictReader(f))
    if not rows:
        raise RuntimeError(f"No rows in {index_path}")
    shard_path = dataset_dir / rows[0]["shard_path"]
    with np.load(shard_path, allow_pickle=False) as z:
        x = z["X"]
        y = z["y"]
        mask = z["valid_mask"]
        print(f"p{patch_size} smoke shard", shard_path.name)
        print("  X", x.shape, x.dtype)
        print("  y", y.shape, y.dtype, "min/mean/max", float(y.min()), float(y.mean()), float(y.max()))
        print("  valid_mask", mask.shape, mask.dtype, "mean", float(mask.mean()))
        print("  split", str(z["split"][0]), "week", str(z["week_key"][0]))

def run_patch_factory() -> None:
    if FAIL_ON_BAD_RAW:
        summary_path = REPORT_DIR / "summary.json"
        if summary_path.exists():
            report = json.loads(summary_path.read_text())
            if int(report.get("bad_count", 0)) > 0:
                raise RuntimeError(
                    f"Raw integrity scan found bad_count={report['bad_count']}. "
                    "Fix raw files or set FAIL_ON_BAD_RAW=False for an engineering-only run."
                )
        else:
            print("No integrity summary found; run scan_raw_integrity() first.")

    tiles = scan_raw_tiles(RAW_DIR)
    print(f"Raw files selected: {len(tiles)}")
    print("Weeks:", sorted({tile.week_key for tile in tiles}))
    print("Tiles:", [(tile.tile_id, tile.tile_key, tile.split) for tile in sorted({t.tile_key: t for t in tiles}.values(), key=lambda x: x.tile_id)])

    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    write_inventory(OUTPUT_ROOT, tiles)
    write_split_manifest(OUTPUT_ROOT, tiles)

    for patch_size in PATCH_SIZES:
        generate_shards_for_patch_size(tiles, patch_size)
        rebuild_index_and_manifest(tiles, patch_size)
        smoke_load(patch_size)

    print("Patch factory complete:", OUTPUT_ROOT)


## Run p128 Patch Factory

Run this only after the integrity scan. With `FAIL_ON_BAD_RAW=True`, the factory stops if the scan found bad raw files.


In [ ]:
run_patch_factory()


## Rebalance p128 Split

This season dataset is for engineering benchmark and Kaggle transfer. It uses sample-balanced tile split based on p128 tile counts. The formal five-year dataset should use the 3+1+1 temporal split instead.


In [ ]:
if True:
    ROOT = OUTPUT_ROOT
    PRIMARY_SIZE = "p128"

    def build_sample_balanced_split(root: Path) -> dict[str, str]:
        df = pd.read_csv(root / PRIMARY_SIZE / "index.csv")
        counts = df.groupby("tile_key")["num_samples"].sum().sort_values(ascending=False)
        print("Primary tile counts:")
        print(counts)
        if len(counts) < 3:
            raise RuntimeError("Need at least 3 sampled tiles for train/val/test split")
        tile_keys = list(counts.index)
        if len(counts) == 3:
            split_by_tile = {tile_keys[0]: "train"}
            split_by_tile[tile_keys[1]] = "val"
            split_by_tile[tile_keys[2]] = "test"
            return split_by_tile
        split_by_tile: dict[str, str] = {}
        for tile_key in tile_keys[:2]:
            split_by_tile[tile_key] = "train"
        split_by_tile[tile_keys[2]] = "test"
        for tile_key in tile_keys[3:]:
            split_by_tile[tile_key] = "val"
        return split_by_tile

    def rewrite_shard_split(shard_path: Path, split: str) -> None:
        with np.load(shard_path, allow_pickle=False) as z:
            arrays = {k: z[k] for k in z.files}
        n = int(arrays["y"].shape[0])
        arrays["split"] = np.asarray([split] * n)
        tmp_path = shard_path.with_suffix(".tmp.npz")
        if tmp_path.exists():
            tmp_path.unlink()
        np.savez_compressed(tmp_path, **arrays)
        tmp_path.replace(shard_path)

    def rebuild_index_and_manifest_for_split(root: Path, size: str) -> None:
        dataset_dir = root / size
        index_path = dataset_dir / "index.csv"
        manifest_path = dataset_dir / "manifest.json"
        rows = [read_shard_summary(dataset_dir, shard_path) for shard_path in sorted((dataset_dir / "shards").glob("*.npz"))]
        df = pd.DataFrame(rows)
        df = df.sort_values(["week_key", "tile_id", "row_start", "shard_path"])
        df.to_csv(index_path, index=False)
        split_counts = df.groupby("split")["num_samples"].sum().to_dict()
        week_counts = df.groupby("week_key")["num_samples"].sum().to_dict()

        with manifest_path.open() as f:
            manifest = json.load(f)
        manifest["split_strategy"] = "sample_balanced_season_from_p128_tile_counts"
        manifest["split_rebalanced_at"] = utc_now()
        manifest["total_shards"] = int(len(df))
        manifest["total_samples"] = int(df["num_samples"].sum())
        manifest["split_counts"] = {k: int(v) for k, v in split_counts.items()}
        manifest["week_counts"] = {k: int(v) for k, v in week_counts.items()}
        write_json(manifest_path, manifest)

    def rewrite_dataset_splits(root: Path, size: str, split_by_tile: dict[str, str]) -> None:
        dataset_dir = root / size
        df = pd.read_csv(dataset_dir / "index.csv")
        for _, row in df.iterrows():
            tile_key = row["tile_key"]
            if tile_key not in split_by_tile:
                continue
            rewrite_shard_split(dataset_dir / row["shard_path"], split_by_tile[tile_key])
        rebuild_index_and_manifest_for_split(root, size)

    def update_split_manifest(root: Path, split_by_tile: dict[str, str]) -> None:
        path = root / "split_manifest.json"
        with path.open() as f:
            manifest = json.load(f)
        manifest["split_strategy"] = "sample_balanced_season_from_p128_tile_counts"
        manifest["split_rebalanced_at"] = utc_now()
        manifest["tile_split"] = split_by_tile
        if "tiles" in manifest:
            for tile in manifest["tiles"]:
                tile_key = tile["tile_key"]
                if tile_key in split_by_tile:
                    tile["split"] = split_by_tile[tile_key]
        write_json(path, manifest)

    split_by_tile = build_sample_balanced_split(ROOT)
    print("\nNew split_by_tile:")
    for k, v in split_by_tile.items():
        print(k, "->", v)

    rewrite_dataset_splits(ROOT, PRIMARY_SIZE, split_by_tile)
    update_split_manifest(ROOT, split_by_tile)

    print("\nRebalanced summary:")
    with (ROOT / PRIMARY_SIZE / "manifest.json").open() as f:
        manifest = json.load(f)
    print("total_shards:", manifest["total_shards"])
    print("total_samples:", manifest["total_samples"])
    print("split_counts:", manifest["split_counts"])
    print("week_counts:", manifest["week_counts"])


## Validate Final p128 Output

This prints final manifest counts and checks train, val, and test splits are present.


In [ ]:
manifest_path = OUTPUT_ROOT / "p128" / "manifest.json"
index_path = OUTPUT_ROOT / "p128" / "index.csv"

with manifest_path.open() as f:
    manifest = json.load(f)

print(json.dumps({
    "total_shards": manifest["total_shards"],
    "total_samples": manifest["total_samples"],
    "split_counts": manifest["split_counts"],
    "week_counts": manifest["week_counts"],
}, indent=2, sort_keys=True))

split_counts = manifest["split_counts"]
missing = [name for name in ("train", "val", "test") if int(split_counts.get(name, 0)) <= 0]
if missing:
    raise RuntimeError(f"Missing non-empty splits: {missing}")

df = pd.read_csv(index_path)
print("index rows:", len(df))
print(df.groupby("split")["num_samples"].sum())


## Optional Kaggle Dataset Upload

This uploads the staged folder directly from Drive. It does not copy the staged dataset into `/content`.

Upload `kaggle.json` to `/content/kaggle.json` before running this cell. Do not commit or share the token.


In [ ]:
# Edit these before running the Kaggle upload commands.
KAGGLE_USERNAME = "your_kaggle_username"
DATASET_SLUG = "ceres-p128-2025w36-w52"
KAGGLE_DATASET_DIR = OUTPUT_ROOT

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle"])

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

if Path("/content/kaggle.json").exists():
    shutil.copy2("/content/kaggle.json", kaggle_dir / "kaggle.json")
    os.chmod(kaggle_dir / "kaggle.json", 0o600)
    print("Configured Kaggle token")
else:
    print("Upload /content/kaggle.json before running Kaggle CLI commands.")

metadata = {
    "title": "Ceres p128 2025W36-W52",
    "id": f"{KAGGLE_USERNAME}/{DATASET_SLUG}",
    "licenses": [{"name": "CC0-1.0"}],
}
metadata_path = KAGGLE_DATASET_DIR / "dataset-metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2) + "\n")
print(metadata_path.read_text())
print("Dataset folder:", KAGGLE_DATASET_DIR)

# First create private dataset:
# !kaggle datasets create -p "$KAGGLE_DATASET_DIR" --private

# Later update existing dataset:
# !kaggle datasets version -p "$KAGGLE_DATASET_DIR" -m "Update p128 2025W36-W52 staged dataset"
